In [ ]:
# --- Imports 

import numpy as np
import pandas as pd
from umap import UMAP
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler
%matplotlib inline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import RandomForestClassifier

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
# --- Loading all the datasets

# Train dataset
train_dataset = pd.read_csv('data/train_radiomics_hipocamp.csv')
# Test dataset
test_dataset = pd.read_csv('data/test_radiomics_hipocamp.csv')

In [ ]:
# --- Exploring the train dataset

print(f"DS SHAPE\n{train_dataset.shape}\n")

print(f"DATA TYPE\n{train_dataset.dtypes.value_counts()}\n")

object_columns = train_dataset.select_dtypes(include='object').columns
print(f"OBJECT COLUMNS\n{object_columns.tolist()}\n")

missing_summary = train_dataset.isnull().sum()
print(f"MISSING VALUES\n{(missing_summary > 0).sum()}\n")

print(train_dataset['Transition'].value_counts(normalize=True))
unique_counts = train_dataset.nunique()

print(f"\nUNIQUE VALUE COLUMN\n{(unique_counts == 1).sum()}")

In [ ]:
# --- Removing columns with irrelevant information

# Getting all columns whose value is an object
obj_col = train_dataset.select_dtypes(include='object').columns

# Storing in a list the columns that have the same number of unique values as the number of rows
# This means that the column is a unique identifier and probably should be dropped
unique_cols = [col for col in obj_col if train_dataset[col].nunique() == train_dataset.shape[0]]

print(f"UNIQUE COLUMNS\n{unique_cols}\n")

# The following features might actually be useful:
cols_to_keep = [
    'diagnostics_Mask-original_BoundingBox',
    'diagnostics_Mask-original_CenterOfMassIndex',
    'diagnostics_Mask-original_CenterOfMass'
]

# removing last 3 columns from the 'unique_cols' - we don't want to drop them
unique_cols = [col for col in unique_cols if col not in cols_to_keep]

# Dropping the columns stored in the unique_cols list
train_dataset.drop(columns=unique_cols, inplace=True)

# Dropping them from the test dataset as well
test_dataset.drop(columns=unique_cols, inplace=True)

pd.set_option('display.max_colwidth', None)
display(train_dataset['diagnostics_Mask-original_BoundingBox'])
display(train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].head())
display(train_dataset['diagnostics_Mask-original_CenterOfMass'].head())
print(f"\nDS SHAPE\n{train_dataset.shape}")

In [ ]:
# --- Checking if 'diagnostics_Mask-original_CenterOfMassIndex' and 'diagnostics_Mask-original_CenterOfMass' are the same
# They are, we can drop one of them

print(f"DS SHAPE\n{train_dataset.shape}\n")
are_equal = ((train_dataset['diagnostics_Mask-original_CenterOfMassIndex'] == train_dataset['diagnostics_Mask-original_CenterOfMass']).all())
print(f"ARE EQUAL: {are_equal}\n")

# They are the same, so we can drop one of them
train_dataset.drop('diagnostics_Mask-original_CenterOfMass', axis=1, inplace=True)
test_dataset.drop('diagnostics_Mask-original_CenterOfMass', axis=1, inplace=True)
print(f"DS SHAPE\n{train_dataset.shape}")

In [ ]:
# --- Transfoming the 'diagnostics_Mask-original_BoundingBox' feature into useful data

print(f"DS SHAPE\n{train_dataset.shape}\n")

# Splitting the string into 6 different columns
# probably x,y,z,width,height,depth -> most common

train_dataset['diagnostics_Mask-original_BoundingBox'] = train_dataset['diagnostics_Mask-original_BoundingBox'].str.replace('(', '').str.replace(')', '')
train_dataset['diagnostics_Mask-original_BoundingBox'] = train_dataset['diagnostics_Mask-original_BoundingBox'].str.split(',').apply(lambda x: [int(i) for i in x])
train_dataset[['x', 'y', 'z', 'width', 'height', 'depth']] = pd.DataFrame(train_dataset['diagnostics_Mask-original_BoundingBox'].tolist(), index=train_dataset.index)
train_dataset.drop('diagnostics_Mask-original_BoundingBox', axis=1, inplace=True)

# Doing the same for the test dataset
test_dataset['diagnostics_Mask-original_BoundingBox'] = test_dataset['diagnostics_Mask-original_BoundingBox'].str.replace('(', '').str.replace(')', '')
test_dataset['diagnostics_Mask-original_BoundingBox'] = test_dataset['diagnostics_Mask-original_BoundingBox'].str.split(',').apply(lambda x: [int(i) for i in x])
test_dataset[['x', 'y', 'z', 'width', 'height', 'depth']] = pd.DataFrame(test_dataset['diagnostics_Mask-original_BoundingBox'].tolist(), index=test_dataset.index)
test_dataset.drop('diagnostics_Mask-original_BoundingBox', axis=1, inplace=True)

# Calculating the volume of the bounding box
train_dataset['volume'] = train_dataset['width'] * train_dataset['height'] * train_dataset['depth']
test_dataset['volume'] = test_dataset['width'] * test_dataset['height'] * test_dataset['depth']
print(f"DS SHAPE\n{train_dataset.shape}")

In [ ]:
# --- Transfoming the 'diagnostics_Mask-original_CenterOfMassIndex' feature into useful data

print(f"DS SHAPE\n{train_dataset.shape}\n")

# Splitting the string into 3 different columns
# x_center, y_center, z_center

train_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.replace('(', '').str.replace(')', '')
train_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.split(',').apply(lambda x: [float(i) for i in x])
train_dataset[['x_center', 'y_center', 'z_center']] = pd.DataFrame(train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].tolist(), index=train_dataset.index)
train_dataset.drop('diagnostics_Mask-original_CenterOfMassIndex', axis=1, inplace=True)

# Doing the same for the test dataset
test_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = test_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.replace('(', '').str.replace(')', '')
test_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = test_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.split(',').apply(lambda x: [float(i) for i in x])
test_dataset[['x_center', 'y_center', 'z_center']] = pd.DataFrame(test_dataset['diagnostics_Mask-original_CenterOfMassIndex'].tolist(), index=test_dataset.index)
test_dataset.drop('diagnostics_Mask-original_CenterOfMassIndex', axis=1, inplace=True)

# Calculating the distance from the center of mass to the bounding box
train_dataset['distance'] = np.sqrt((train_dataset['x'] - train_dataset['x_center'])**2 + (train_dataset['y'] - train_dataset['y_center'])**2 + (train_dataset['z'] - train_dataset['z_center'])**2)
test_dataset['distance'] = np.sqrt((test_dataset['x'] - test_dataset['x_center'])**2 + (test_dataset['y'] - test_dataset['y_center'])**2 + (test_dataset['z'] - test_dataset['z_center'])**2)
print(f"DS SHAPE\n{train_dataset.shape}")

In [ ]:
# --- Dropping constant features - features that have the same value for all rows
train_dataset = train_dataset.loc[:, (train_dataset != train_dataset.iloc[0]).any()]

# Dropping the same columns on the test dataset
test_dataset = test_dataset[train_dataset.drop(columns='Transition').columns]

print(f"DS SHAPE\n{train_dataset.shape}")

In [ ]:
# --- Checking all unique values in the Transition column
train_dataset['Transition'].value_counts()
# Dataset is extremely UNBALANCED

In [ ]:
# --- Using encoding to transform the target columns into numeric values

replace_map = {'Transition': {'CN-CN': 1, 'AD-AD': 2, 'CN-MCI': 3, 'MCI-AD': 4, 'MCI-MCI': 5}}
print(f"TRANSITION BEFORE ENCODING:\n{train_dataset['Transition'].head()}\n")
train_dataset = train_dataset.replace(replace_map)
test_dataset = test_dataset.replace(replace_map)
print(f"TRANSITION AFTER ENCODING:\n{train_dataset['Transition'].head()}")

In [ ]:
train_dataset['Transition'].value_counts()

In [ ]:
# --- Dropping non-target columns with the same info (correlation = 1 between them)

print(f"DS SHAPE\n{train_dataset.shape}\n")
target_column = 'Transition'
# Calculating the correlation matrix (excluding the target column)
train_dataset_tmp = train_dataset
corr_matrix_train = train_dataset_tmp.drop(columns=[target_column]).corr().abs()

# Upper triangle matrix of correlations
upper = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))

# Finding index of feature columns with correlation equal to 1
threshold = 1.0
to_drop = [column for column in upper.columns if any(upper[column] >= threshold)] # for some reason there are columns with correlation > 1.0
print("DROPPING " + str(len(to_drop)) + " COLUMNS\n")

# Dropping the columns
train_dataset = train_dataset.drop(columns=to_drop)
test_dataset = test_dataset.drop(columns=to_drop)
print(f"DS SHAPE\n{train_dataset.shape}")

In [14]:
## --- Dropping columns with low correlation with the target column (<0.005)

#print(f"DS SHAPE\n{train_dataset.shape}\n")
#corr_matrix = train_dataset.corr()
#corr_target = corr_matrix['Transition'].abs()
#uncorr_cols = []
#for i in range(len(corr_target)):
#  if corr_target.iloc[i] < 0.005:
#    uncorr_cols.append(corr_matrix.columns[i])
#
## Dropping the columns stored in the uncorr_cols list
#n_cols_before = len(train_dataset.columns)
#train_dataset.drop(uncorr_cols, axis=1, inplace=True)
#test_dataset.drop(uncorr_cols, axis=1, inplace=True)
#n_cols_after = len(train_dataset.columns)
#
#print("DROPPED " + str(n_cols_before-n_cols_after) + " COLUMNS!\n")
#print(f"DS SHAPE\n{train_dataset.shape}")

In [ ]:
# --- Checking Age column
print(train_dataset['Age'].hist())
plt.show()
# Not a normal distribution

In [ ]:
# --- Checking Sex column
print(train_dataset['Sex'].value_counts(normalize=True))
# Not quite balanced

In [ ]:
# --- Understanding the relationship between Age and Transition
sns.catplot(x="Transition", y="Age", data=train_dataset, kind="box", aspect=1.5)
plt.title("Boxplot for Transition vs Age")
plt.show()

In [ ]:
# --- Treating outliers
# Outliers should be converted to the closest bound

print(f"DS SHAPE\n{train_dataset.shape}\n")

changes_count = 0

for col in train_dataset.columns:
    if col not in ['Transition', 'ID']:
        median = train_dataset[col].median()
        q1 = train_dataset[col].quantile(0.25)
        q3 = train_dataset[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Count outliers before replacing them
        changes_count += ((train_dataset[col] < lower_bound) | (train_dataset[col] > upper_bound)).sum()
        changes_count += ((test_dataset[col] < lower_bound) | (test_dataset[col] > upper_bound)).sum()

        # Replace outliers with the median
        # train_dataset[col] = train_dataset[col].apply(lambda x: median if x < lower_bound or x > upper_bound else x)
        # test_dataset[col] = test_dataset[col].apply(lambda x: median if x < lower_bound or x > upper_bound else x)

        # Replace outliers with the closest bound
        train_dataset[col] = train_dataset[col].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)
        test_dataset[col] = test_dataset[col].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)

print(f"TOTAL NUMBER OF CHANGES MADE: {changes_count}\n")
print(f"DS SHAPE\n{train_dataset.shape}")

In [ ]:
# --- Dropping more constant features
# Outlier treatment creates more constant features

columns_to_exclude = train_dataset.columns[(train_dataset == train_dataset.iloc[0]).all()]
print(f"COLUMNS TO BE EXCLUDED:\n{columns_to_exclude.tolist()}\n")
train_dataset = train_dataset.drop(columns=columns_to_exclude)

In [20]:
# --- Standardizing data
# Using the same scaler for both datasets

scaler = StandardScaler()
columns_to_scale = train_dataset.drop(columns=['Transition']).columns
train_dataset[columns_to_scale] = scaler.fit_transform(train_dataset[columns_to_scale])
test_dataset[columns_to_scale] = scaler.transform(test_dataset[columns_to_scale])

In [21]:
# Add an 'oversampled' column to the dataset with 'NO' values
# This will be used to guarantee oversampling data is not used in testing
oversampled_column = pd.Series('NO', index=train_dataset.index, name='oversampled')
train_dataset = pd.concat([train_dataset, oversampled_column], axis=1)

In [ ]:
# Checking number of samples per class
train_dataset['Transition'].value_counts()

In [ ]:
# --- Oversampling data
### Oversampling should be done after standardizing but before feature selection and dimensionality reduction

from imblearn.over_sampling import BorderlineSMOTE
from imblearn.over_sampling import ADASYN
from imblearn.over_sampling import SVMSMOTE
from imblearn.over_sampling import KMeansSMOTE

print(f"DS SHAPE\n{train_dataset.shape}\n")
print("CLASS DISTRIBUTION BEFORE SMOTE OVERSAMPLING:\n")
print(train_dataset['Transition'].value_counts())

# Split the dataset into features and target
X = train_dataset.drop(columns=['Transition', 'oversampled'])
y = train_dataset['Transition']

# Identify the majority class count (class '1')
majority_class_count = y.value_counts().loc[1]

# Define the sampling strategy
sampling_strategy = {1: 96,
                     2: 96,
                     3: 31,
                     4: 96,
                     5: 96
}



# Apply SMOTE with the defined strategy
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=123)
#smote = BorderlineSMOTE(sampling_strategy=sampling_strategy, kind='borderline-1')
#smote = BorderlineSMOTE(sampling_strategy=sampling_strategy, kind='borderline-2')
#smote = SVMSMOTE(sampling_strategy=sampling_strategy)

X_smote, y_smote = smote.fit_resample(X, y)

# Create a new dataframe with the oversampled data
oversampled_data = pd.concat([X_smote, y_smote], axis=1)
oversampled_data.columns = list(X.columns) + ['Transition']

# Mark the oversampled rows with 'YES' in the 'oversampled' column
oversampled_data['oversampled'] = ['YES' if idx >= len(X) else 'NO' for idx in range(len(X_smote))]

# Concatenate the new oversampled data with the original data
train_dataset = pd.concat([train_dataset, oversampled_data[oversampled_data['oversampled'] == 'YES']], ignore_index=True)

# Display the final class distribution for verification
print("\nCLASS DISTRIBUTION AFTER SMOTE OVERSAMPLING:\n")
print(train_dataset['Transition'].value_counts())

# --- Getting the number of normal and oversampled data
train_dataset['oversampled'].value_counts()

In [ ]:
# --- Feature selection with ANOVA

print(f"DS SHAPE\n{train_dataset.shape}\n")
X = train_dataset.drop(columns=['Transition', 'oversampled'])
y = train_dataset['Transition']

f, p = f_classif(X, y)
feature_importances = pd.Series(f, index=X.columns)
print(f"MOST IMPORTANT FEATURES\n\n{feature_importances.sort_values(ascending=False).head(10)}\n")
num_features_to_keep = 200
features_to_keep = X.columns[f.argsort()[-num_features_to_keep:]]
# Keeping the selected features on the train dataset + transition column + oversampled column
X = X[list(features_to_keep)]
X = pd.concat([X, train_dataset[['Transition', 'oversampled']]], axis=1)
train_dataset = X

# Dropping the non-selected columns from the test dataset
test_dataset = test_dataset[features_to_keep]
print(f"DS SHAPE\n{X.shape}")

In [25]:
# --- Dimensionality reduction with PCA

X = train_dataset.drop(columns=['Transition', 'oversampled'])
y = train_dataset['Transition']
pca = PCA(n_components=150)
pca.fit(X)
data = pca.transform(X)
train_dataset_no_target = pd.DataFrame(data)
train_dataset = pd.concat([train_dataset_no_target, train_dataset['Transition'], train_dataset['oversampled']], axis=1)

data_test = pca.transform(test_dataset)
test_dataset = pd.DataFrame(data_test)

In [ ]:
# --- Dimensionlity Reduction with UMAP

'''
X = train_dataset.drop(columns=['Transition', 'oversampled'])
y = train_dataset['Transition']
umap = UMAP(
    n_components=50, 
    n_neighbors=50,
    min_dist=0.5,
    metric='euclidean',
    random_state=42,
    )

X_umap = umap.fit_transform(X)
train_dataset_no_target = pd.DataFrame(X_umap)
train_dataset = pd.concat([train_dataset_no_target, train_dataset['Transition'], train_dataset['oversampled']], axis=1)

data_test = umap.transform(test_dataset)
test_dataset = pd.DataFrame(data_test)
'''

In [27]:
## --- Checking feature importance for the engineered features
#
#X = train_dataset.drop(columns=['Transition', 'oversampled'])
#y = train_dataset['Transition']
#
## Use RandomForest for feature importance
#rf = RandomForestClassifier(n_estimators=100, random_state=123)
#rf.fit(X, y)
#importances = rf.feature_importances_
#importance_indices = importances.argsort()[::-1]
#top_features = X.columns[importance_indices[:20]]
#top_importances = importances[importance_indices[:20]]
#features_to_keep = X.columns[importance_indices]
#
#feature_names = ['volume', 'distance', 'x', 'y', 'z', 'width', 'height', 'depth', 'x_center', 'y_center', 'z_center']
#
#print("20 MORE IMPORTANT FEATURES:\n")
#for i, (feature, importance) in enumerate(zip(top_features, top_importances), start=1):
#  print(f"{i}. {feature} - Importance: {importance:.4f}")
#
#print("\nOUR INTRODUCED FEATURES AND IMPORTANCES:\n")
#for feature in feature_names:
#    if feature in X.columns:
#        feature_index = list(X.columns).index(feature)
#        rank = np.where(importance_indices == feature_index)[0][0] + 1  # Ranking começa em 1
#        importance_value = importances[feature_index]  # Valor exato da importância
#        print(f"Ranking of '{feature}': {rank} out of {len(importance_indices)} (Importance: {importance_value:.4f})")
#    else:
#        print(f"Feature '{feature}' is not present in the dataset.")

In [ ]:
# Guaranteeing all columns are shown on .head
pd.set_option('display.max_columns', None)

train_dataset.head()

In [ ]:
test_dataset.head()

In [30]:
# --- Saving the treated datasets
train_dataset.to_csv('data/train_dataset.csv', index=False)
test_dataset.to_csv('data/test_dataset.csv', index=False)